# DeepFace による顔認識 (face recognition)

顔認識のタスク（判別したいこと）をさらに分けると下記のようなものがある．
- 顔が誰かを判定 （顔認証，不審者検出，etc.）
  - 予め，どの顔が誰かを学習しておく必要がある
- 画像の中での顔の領域を検出 （カメラアプリ，人物追跡，人数カウント, etc.）
- 顔の内容を判定 （性別，年齢，表情，etc.）

DeepFace や OpenFace, face-api.js 等，既に大量の顔で学習済み (pretrained) のモデルが数多くある．

ここでは，DeepFace を使って，顔の性別，年齢，表情を識別するシステムを構築する．

### まず，DeepFace を pip で colab 上にインストールする

pip は，python のライブラリの管理を行うツール．  
（他に anaconda も良く使われるが, ここでは pip を使う）

### pip は，UNIX (Linux) のコマンドなので，Jupyter からは，! を頭に付けてコマンドを実行する．

In [ ]:
!pip install deepface

In [ ]:
import numpy as np
from deepface import DeepFace
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import requests
from PIL import Image
from io import BytesIO
from IPython.core.display import HTML
import base64
%matplotlib inline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### カメラアプリを作る．

camera_on() という関数で，カメラアプリを作る．  

アプリ（というほどではないが）は，HTML (とスタイルシート (css) = style タグ内)で表示の配置が定義され，カメラ表示やボタン動作は，JavaScript ( script タグ内 )で定義されている．  

USB カメラから，HTML の JavaScript の navigator.mediaDevices.getUserMedia 関数で，カメラ映像を HTML 上の canvas に表示し，「保存！」ボタンが押されたら，canvas.toDataURL('image/png') 関数で，python の data 変数に画像にして格納する．



In [ ]:
def camera_on():
  from IPython.display import HTML, Image
  from google.colab.output import eval_js
  from base64 import b64decode
  from io import BytesIO
  from PIL import Image

  canvas_html = """
  <style type="text/css">
  #video { border: 1px solid yellow; width: 640px; height: 480px;}
  </style>

  <button>保存！</button>
  <video id="video"></video>
  <canvas width=640 height=480></canvas>

  <script>
  var canvas = document.querySelector('canvas')
  var button = document.querySelector('button')
  const video = document.querySelector('#video');

  navigator.mediaDevices.getUserMedia({ video: true, audio: false })
          .then((stream) => {
              video.srcObject = stream;
              video.play();
          })

  var data = new Promise(resolve=>{
    button.onclick = ()=>{
      const context = canvas.getContext("2d");
      context.drawImage(video, 0, 0, canvas.width, canvas.height);
      resolve(canvas.toDataURL('image/png'))
    }
  })
  </script>
  """

  # HTML表示
  display(HTML(canvas_html))
  data = eval_js("data")
  # ボタンが押された後，canvasデータがbinaryに入る (BASE64エンコードで来るのでデコード)
  binary = b64decode(data.split(',')[1])
  img = Image.open(BytesIO(binary))
  # 背景が透明なため，白背景画像 background を作り書いた文字を重ね，img に入れる
  background = Image.new("RGB", img.size, (255, 255, 255))
  background.paste(img, img)
  img = background
  return img

## 注意！！
- 最初，ブラウザからカメラ起動の許可が聞かれたら許可する．
- 「VMWare virtual」等と表示されているカメラは選択しない．
- 1回目は映像取得に失敗するかも知れないので，停止ボタンを押し，再度セル実行する．
- 黄色の枠内に映像が表示されれば良い．

In [ ]:
img = camera_on()

### 画像の確認

In [ ]:
display(img)

## 画像を Google Drive から読み込む

In [ ]:
img = Image.open("/content/drive/MyDrive/lab2.png")
background = Image.new("RGB", img.size, (255, 255, 255))
background.paste(img, img)
img = background

## 画像を URL から読み込む

In [ ]:
img_url = ""
img = Image.open(BytesIO(requests.get(img_url).content))
background.paste(img, img)
img = background

### DeepFace は，NumPy 配列 (array) 形式で画像を受け取るので，image 形式から変換する．

In [ ]:
img_np = np.asarray(img)

In [ ]:
face_result = DeepFace.analyze(img_np)

### 顔認識の解析結果を見てみる

DeepFace.analyze での解析結果，face_result を見る．  

これは，Python の list 形式になっていて，かつ入れ子（list の要素に list がある）になっている．list は辞書（他言語では，連想配列やハッシュと呼ぶことも）になっていて，例えば，

```python
face_result[0]['dominant_emotion']
face_result[0]['gender']['Woman']
```
のようにして値を取り出せる（1行目はメインの表情，2行目は性別のうち Woman の度合い）．他にも region で顔の画像内での位置等が分かる．


In [ ]:
face_result

In [ ]:
# 顔認識の結果を可視化

顔認識の結果を可視化してみる．

face_result から，顔の座標，年齢，メインの性別，メインの表情を取り出し，顔の画像を表示して，顔の座標に四角の枠 (Rectangle) を，男性なら青，女性なら赤で表示し，その上に，年齢，性別，表情を表示する．


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
plt.figure(figsize=(8, 8))
ax = plt.imshow(img)
for face in face_result:
  fleft = face['region']['x']
  ftop = face['region']['y']
  fwidth = face['region']['w']
  fheight = face['region']['h']

  fage = face['age']
  fgen = face['dominant_gender']
  femo = face['dominant_emotion']
  fcol = 'blue' if fgen=='Man' else 'red'
  p = patches.Rectangle( (fleft, ftop), fwidth, fheight, fill=False, linewidth=2, color=fcol)
  ax.axes.add_patch(p)
  plt.text(fleft, ftop,"%s, %s %s"%( fage ,fgen, femo),  color='white')
_ = plt.axis("off")